In [2]:
import torch

# Load the encoded dataset
data = torch.load('workspace/experiments/sphere-small-small-cifar-10-32px/encoding/encoded_dataset.pt')

# Check what's in the data
print(data.keys())
# Output: dict_keys(['encodings', 'labels', 'split_ids', 'split_names'])

# ============================================================
# STRUCTURE OF THE DATA
# ============================================================

# 1. Encodings: shape [60000, 256, 4]
#    - 60000 total samples (50000 train + 10000 test)
#    - 256 tokens per sample
#    - 4 dimensions per token
encodings = data['encodings']
print(f"Encodings shape: {encodings.shape}")
print(f"Data type: {encodings.dtype}")  # torch.float32
print(f"Device: {encodings.device}")    # cpu
print(f"Memory: {encodings.element_size() * encodings.nelement() / 1e9:.2f} GB")
# Output:
# Encodings shape: torch.Size([60000, 256, 4])
# Data type: torch.float32
# Device: cpu
# Memory: 0.96 GB

# ============================================================
# 2. Labels: shape [60000]
#    - Class label (0-9) for each sample
labels = data['labels']
print(f"Labels shape: {labels.shape}")
print(f"Unique classes: {torch.unique(labels).tolist()}")  # [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
print(f"Class distribution: {torch.bincount(labels).tolist()}")
# Output:
# Labels shape: torch.Size([60000])
# Unique classes: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
# Class distribution: [6000, 6000, 6000, 6000, 6000, 6000, 6000, 6000, 6000, 6000]

# ============================================================
# 3. Split IDs: shape [60000]
#    - 0 = train split
#    - 1 = test split
split_ids = data['split_ids']
print(f"Split IDs shape: {split_ids.shape}")
print(f"Split distribution: {torch.bincount(split_ids).tolist()}")
# Output:
# Split IDs shape: torch.Size([60000])
# Split distribution: [50000, 10000]  # 50000 train, 10000 test

# ============================================================
# 4. Split names: mapping for split IDs
split_names = data['split_names']
print(f"Split names: {split_names}")  # ['train', 'test']

# ============================================================
# HOW TO USE FOR FLOW MATCHING
# ============================================================

# Method 1: Use all data
all_encodings = data['encodings']  # [60000, 256, 4]
all_labels = data['labels']        # [60000]

# Method 2: Split by train/test
train_mask = data['split_ids'] == 0
test_mask = data['split_ids'] == 1

train_encodings = data['encodings'][train_mask]  # [50000, 256, 4]
train_labels = data['labels'][train_mask]        # [50000]

test_encodings = data['encodings'][test_mask]    # [10000, 256, 4]
test_labels = data['labels'][test_mask]          # [10000]

print(f"Train split: {train_encodings.shape}, Test split: {test_encodings.shape}")
# Output: Train split: torch.Size([50000, 256, 4]), Test split: torch.Size([10000, 256, 4])

# Method 3: Further split train into train/val
num_train = train_encodings.shape[0]
val_ratio = 0.2
num_val = int(num_train * val_ratio)

indices = torch.randperm(num_train)
train_idx = indices[num_val:]
val_idx = indices[:num_val]

final_train_encodings = train_encodings[train_idx]  # [40000, 256, 4]
final_train_labels = train_labels[train_idx]        # [40000]

final_val_encodings = train_encodings[val_idx]      # [10000, 256, 4]
final_val_labels = train_labels[val_idx]            # [10000]

print(f"Final: Train [{final_train_encodings.shape}], Val [{final_val_encodings.shape}], Test [{test_encodings.shape}]")
# Output: Final: Train [torch.Size([40000, 256, 4])], Val [torch.Size([10000, 256, 4])], Test [torch.Size([10000, 256, 4])]

# Method 4: Filter by class
class_0_mask = data['labels'] == 0
class_0_encodings = data['encodings'][class_0_mask]  # [6000, 256, 4]
print(f"Class 0 encodings: {class_0_encodings.shape}")
# Output: Class 0 encodings: torch.Size([6000, 256, 4])

# ============================================================
# CREATING DATALOADERS FOR FLOW MATCHING
# ============================================================

from torch.utils.data import TensorDataset, DataLoader

# Create a dataset
train_dataset = TensorDataset(final_train_encodings, final_train_labels)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)

# Iterate
for batch_encodings, batch_labels in train_loader:
    print(f"Batch shape: {batch_encodings.shape}, Labels: {batch_labels.shape}")
    # Output: Batch shape: torch.Size([128, 256, 4]), Labels: torch.Size([128])
    break

dict_keys(['encodings', 'labels', 'split_ids', 'split_names'])
Encodings shape: torch.Size([60000, 256, 4])
Data type: torch.float32
Device: cpu
Memory: 0.25 GB
Labels shape: torch.Size([60000])
Unique classes: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Class distribution: [6000, 6000, 6000, 6000, 6000, 6000, 6000, 6000, 6000, 6000]
Split IDs shape: torch.Size([60000])
Split distribution: [50000, 10000]
Split names: ['train', 'test']
Train split: torch.Size([50000, 256, 4]), Test split: torch.Size([10000, 256, 4])
Final: Train [torch.Size([40000, 256, 4])], Val [torch.Size([10000, 256, 4])], Test [torch.Size([10000, 256, 4])]
Class 0 encodings: torch.Size([6000, 256, 4])
Batch shape: torch.Size([128, 256, 4]), Labels: torch.Size([128])


/tmp/ipykernel_14630/3013578662.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load('workspace/experiments/sphere-small-small-cifar-10-32px/encoding/encode

In [3]:
# ============================================================
# INSPECT FIRST 10 SAMPLES
# ============================================================

print("First 10 samples from the dataset:\n")

for i in range(10):
    enc = data['encodings'][i]
    lbl = data['labels'][i].item()
    split_id = data['split_ids'][i].item()
    split_name = data['split_names'][split_id]
    
    print(f"Sample {i}:")
    print(f"  Label: {lbl} (class {lbl})")
    print(f"  Split: {split_name} (split_id={split_id})")
    print(f"  Encoding shape: {enc.shape}")
    print(f"  Encoding min/max: {enc.min():.4f} / {enc.max():.4f}")
    print(f"  Encoding mean/std: {enc.mean():.4f} / {enc.std():.4f}")
    print(f"  First token (5 dims): {enc[0, :5]}")
    print()


First 10 samples from the dataset:

Sample 0:
  Label: 6 (class 6)
  Split: train (split_id=0)
  Encoding shape: torch.Size([256, 4])
  Encoding min/max: -1.5760 / 1.5651
  Encoding mean/std: -0.0489 / 0.5498
  First token (5 dims): tensor([ 1.4832,  0.1805,  0.1403, -0.0188])

Sample 1:
  Label: 9 (class 9)
  Split: train (split_id=0)
  Encoding shape: torch.Size([256, 4])
  Encoding min/max: -1.5319 / 1.6162
  Encoding mean/std: -0.0118 / 0.6519
  First token (5 dims): tensor([-0.0287,  0.4000, -0.3501,  0.4091])

Sample 2:
  Label: 9 (class 9)
  Split: train (split_id=0)
  Encoding shape: torch.Size([256, 4])
  Encoding min/max: -1.4703 / 1.3482
  Encoding mean/std: -0.0371 / 0.5244
  First token (5 dims): tensor([-0.7679, -0.2749, -0.3764,  0.3104])

Sample 3:
  Label: 4 (class 4)
  Split: train (split_id=0)
  Encoding shape: torch.Size([256, 4])
  Encoding min/max: -1.6135 / 1.3242
  Encoding mean/std: 0.0050 / 0.4334
  First token (5 dims): tensor([ 0.9130, -0.5050,  0.4538, -0.6